<u><b><h1 style="text-align:center; line-height:25px; color:#000000; background:#EFEFEF; border: 1px solid #FF6B6B ; padding:20px;">Extracting Prevalent Topics from Citizens Complaints using NLP</h1></b></u>
<u><h2 style="text-align:center">3. Vectorization step</h2></u>
**Course:** DLBDSEDA02 – Project: Data Analysis  
**Tools**:  
**Dataset:** <a href="https://www.kaggle.com/datasets/xjoury/customer-complaints-sentiment-and-priority-dataset">Customer Complaints Sentiment and Priority Dataset</a>  
**<a href="https://github.com/davidlupau/complaint-topic-modeling">GitHub repository</a>**

<b><h2 style="padding: 10px; border-left: 3px solid #FF6B6B;">Setup & Imports</h2></b>

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir()
)
sys.path.append(str(PROJECT_ROOT / "src"))

from vectorization import (
    build_tfidf_vectorizer,
    build_count_vectorizer,
    summarize_vectorizer,
    top_terms_by_document_frequency,
    save_vectorization_artifacts,
)

print("Import successful")

Import successful


### Loading clean dataset

In [2]:
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "complaints_clean.csv")
print(f"Loaded {df.shape[0]} cleaned complaints")

Loaded 1745 cleaned complaints


---

<b><h2 style="padding: 10px; border-left: 3px solid #FF6B6B;">Pipeline A — TF-IDF (for LSA)</h2></b>

LSA (via truncated SVD) decomposes a term-weighted matrix, so it needs TF-IDF weights rather than raw counts: TF-IDF downweights terms that are common across many documents and upweights terms that are distinctive to a given complaint, which is exactly the signal LSA's singular value decomposition relies on to separate topics.

### Fitting TfidfVectorizer

`02_preprocessing.ipynb`'s post-cleaning vocabulary check found that generic stopword removal doesn't catch domain-specific words that are ubiquitous across complaints regardless of topic (e.g. "account", "credit"), and that lemmatization/tokenization leave behind rare one-off noise (typos, fragments). `min_df=5` drops terms appearing in fewer than 5 documents, filtering that rare noise; `max_df=0.5` drops terms appearing in more than half the corpus, targeting exactly the ubiquitous-term problem the vocabulary check flagged. `ngram_range=(1, 1)` and `max_features=None` keep the vocabulary to unigrams with no artificial cap, per the conception document's scope (no bigrams, no custom curation).

In [3]:
tfidf_vectorizer, tfidf_matrix = build_tfidf_vectorizer(df["cleaned_complaint"])

tfidf_summary = summarize_vectorizer(tfidf_vectorizer, tfidf_matrix)
top_tfidf_terms = top_terms_by_document_frequency(tfidf_vectorizer, tfidf_matrix, top_n=20)
print("\nTop 20 terms by document frequency:")
print(top_tfidf_terms)

Fitted TfidfVectorizer on 1745 documents
Vocabulary size: 2600
Matrix shape: 1745 documents x 2600 terms
Sparsity: 97.8425%

Top 20 terms by document frequency:
term
credit         753
time           663
would          605
payment        577
received       567
called         525
bank           521
told           520
day            495
company        464
never          454
back           447
get            447
call           441
also           439
money          433
loan           431
information    427
report         424
sent           420
Name: doc_frequency, dtype: int64


Fitting produced a vocabulary of **2,600 terms** over a **1,745 × 2,600** matrix, **97.84% sparse** — as expected for a document-term matrix, since most complaints only use a small fraction of the vocabulary.

Checking whether `max_df=0.5` actually removed the ubiquitous terms flagged earlier: it partially did. `"account"` appeared in 52.44% of documents (915 of 1,745) — just above the 50% threshold — and was correctly excluded from the vocabulary entirely. However, `"credit"` (43.15%), `"would"` (34.67%) and `"bank"` (29.86%) all sit below the threshold and remain in the vocabulary; they still dominate the top-20 document-frequency list above. So `max_df=0.5` filtered the single most extreme case ("account") but did not remove every ubiquitous domain term — the assumption that this parameter alone would fully solve the ubiquitous-term problem does not hold. This is worth keeping in mind when interpreting LSA components: components dominated by "credit", "would" or "bank" may reflect corpus-wide language rather than a distinct topic.

---

<b><h2 style="padding: 10px; border-left: 3px solid #FF6B6B;">Pipeline B — CountVectorizer (for LDA)</h2></b>

LDA models documents as a mixture of topics generated from raw word counts (a Dirichlet-multinomial process), not re-weighted TF-IDF scores — feeding it IDF-weighted values would violate that generative assumption. `CountVectorizer` is fit here with the identical vocabulary-shaping parameters as Pipeline A (`min_df`, `max_df`, `ngram_range`, `max_features`), so both pipelines model the same vocabulary and differ only in how term importance is weighted.

In [4]:
count_vectorizer, count_matrix = build_count_vectorizer(df["cleaned_complaint"])

count_summary = summarize_vectorizer(count_vectorizer, count_matrix)

vocab_matches = set(tfidf_vectorizer.vocabulary_) == set(count_vectorizer.vocabulary_)
print(f"\nVocabulary matches Pipeline A: {vocab_matches}")

Fitted CountVectorizer on 1745 documents
Vocabulary size: 2600
Matrix shape: 1745 documents x 2600 terms
Sparsity: 97.8425%

Vocabulary matches Pipeline A: True


CountVectorizer produced the identical **2,600-term vocabulary** over the same **1,745 × 2,600** shape and **97.84% sparsity** as Pipeline A, and the vocabulary-match check confirms the two vectorizers' term sets are exactly equal. This is expected: `min_df`, `max_df`, `ngram_range` and `max_features` are the only parameters that determine *which* terms enter the vocabulary, and both pipelines use identical values for all four — `use_idf`/`smooth_idf`/`sublinear_tf` only affect the weighting of matrix entries, not vocabulary membership. The matrices themselves differ (raw integer counts here vs. TF-IDF floats in Pipeline A), which is exactly the intended difference for feeding LDA vs. LSA respectively.

---

<b><h2 style="padding: 10px; border-left: 3px solid #FF6B6B;">Saving vectorization artifacts</h2></b>

### Persisting both pipelines

Both fitted vectorizers and their document-term matrices are persisted to `data/vectorized/` so `04_topic_modeling.ipynb` can load them directly and fit LSA/LDA without re-running vectorization from scratch.

In [5]:
save_vectorization_artifacts(tfidf_vectorizer, tfidf_matrix, "tfidf")
save_vectorization_artifacts(count_vectorizer, count_matrix, "count")

Saved vectorizer to: /Users/davidlupau/Documents/Filen/_IU/6th semester/32. Project Data Analysis/complaint-topic-modeling/data/vectorized/tfidf_vectorizer.joblib
Saved matrix to: /Users/davidlupau/Documents/Filen/_IU/6th semester/32. Project Data Analysis/complaint-topic-modeling/data/vectorized/tfidf_matrix.npz


Saved vectorizer to: /Users/davidlupau/Documents/Filen/_IU/6th semester/32. Project Data Analysis/complaint-topic-modeling/data/vectorized/count_vectorizer.joblib
Saved matrix to: /Users/davidlupau/Documents/Filen/_IU/6th semester/32. Project Data Analysis/complaint-topic-modeling/data/vectorized/count_matrix.npz


Both vectorizers (`tfidf_vectorizer.joblib`, `count_vectorizer.joblib`) and both matrices (`tfidf_matrix.npz`, `count_matrix.npz`) are now saved in `data/vectorized/`, ready to be loaded in the next notebook without repeating the fitting step above.